# RAG-NetTwin: Embedding & Reranker Benchmark Suite

This notebook benchmarks various embedding and reranker models for network engineering documentation retrieval. Optimized for Google Colab with GPU acceleration.

In [ ]:
!apt-get install -y ffmpeg --quiet
!pip install "fsspec<=2025.3.0" "rich<14" "transformers>=4.44.0" "opentelemetry-api<1.39.0" "opentelemetry-sdk<1.39.0" chromadb rank-bm25 sentence-transformers==3.3.1 --quiet

In [ ]:
import json
import time
import numpy as np
import torch
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import re
from rich.console import Console
from rich.table import Table

console = Console()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
console.print(f"[bold green]Using device: {DEVICE}[/bold green]")

# --- UTILS ---
def tokenize(text):
    return re.findall(r"\w+", text.lower())

def calculate_mrr(expected_id, retrieved_ids):
    try:
        rank = retrieved_ids.index(expected_id) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0

def calculate_ndcg(expected_id, retrieved_ids):
    try:
        rank = retrieved_ids.index(expected_id) + 1
        return 1.0 / np.log2(rank + 1)
    except ValueError:
        return 0.0

# --- BENCHMARK DATA ---
RETRIEVAL_BENCHMARK = [
    {"query": "LISP site registration information show command", "vendor": "cisco", "expected_id": "chunk_000050"},
    {"query": "display LISP IPv4 address-family packet count statistics", "vendor": "cisco", "expected_id": "chunk_000051"},
    {"query": "show lisp prefix-list information", "vendor": "cisco", "expected_id": "chunk_000052"},
    {"query": "how to set the LISP map-cache limit to 5000 entries", "vendor": "cisco", "expected_id": "chunk_000053"},
    {"query": "show ospf database external detail command on junos", "vendor": "juniper", "expected_id": "chunk_000105"},
    {"query": "difference between LISP proxy-itrs and proxy-etrs implementation", "vendor": "cisco", "expected_id": "chunk_000054"},
    {"query": "troubleshoot lisp registration failures and timeout events", "vendor": "cisco", "expected_id": "chunk_000055"},
    {"query": "define a routing policy term with community and local-preference", "vendor": "juniper", "expected_id": "chunk_000110"},
    {"query": "catalyst 9300 stackwise-virtual configuration requirements", "vendor": "cisco", "expected_id": "chunk_000200"},
    {"query": "show lisp site detail vs show lisp site summary", "vendor": "cisco", "expected_id": "chunk_000056"},
]

# Mock chunk data for Colab if file doesn't exist
try:
    with open("all_chunks.json", "r") as f:
        ALL_CHUNKS = json.load(f)
except FileNotFoundError:
    console.print("[bold red]all_chunks.json not found. Creating dummy data for testing purposes.[/bold red]")
    ALL_CHUNKS = []
    for test in RETRIEVAL_BENCHMARK:
        ALL_CHUNKS.append({"id": test["expected_id"], "text": f"{test['vendor'].capitalize()} command: {test['query']}. Details...", "vendor": test["vendor"]})
    # Add noise
    for i in range(1000):
        ALL_CHUNKS.append({"id": f"noise_{i}", "text": f"Random networking info chunk {i}. BGP OSPF RIP LISP...", "vendor": "cisco" if i % 2 == 0 else "juniper"})

## Part 1: Embedding Model Comparison
This compares the base retrievers.

In [ ]:
def run_embedding_test(model_name):
    console.print(f"\n[bold yellow]Testing Embedding Model: {model_name}[/bold yellow]")
    
    # Load Model
    model = SentenceTransformer(model_name, device=DEVICE, trust_remote_code=True)
    
    # Setup Chroma
    client = chromadb.EphemeralClient()
    collection = client.create_collection(name="embed_test", metadata={"hnsw:space": "cosine"})
    
    # Filter & Index
    expected_ids = {test["expected_id"] for test in RETRIEVAL_BENCHMARK}
    relevant_chunks = [c for c in ALL_CHUNKS if c["id"] in expected_ids]
    noise_chunks = [c for c in ALL_CHUNKS if c["id"] not in expected_ids][:500]
    chunks = relevant_chunks + noise_chunks
    
    texts = [c["text"] for c in chunks]
    ids = [c["id"] for c in chunks]
    metadatas = [{"vendor": c["vendor"]} for c in chunks]
    
    # Handle Granite instructions
    is_granite = "granite" in model_name.lower()
    doc_prefix = "Instruction: Given a network documentation chunk, extract and embed it for retrieval.\nChunk: " if is_granite else ""
    query_prefix = "Instruction: Given a network engineering query, retrieve the most relevant command documentation.\nQuery: " if is_granite else ""
    
    processed_texts = [doc_prefix + t for t in texts]
    embeddings = model.encode(processed_texts, normalize_embeddings=True)
    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings.tolist())
    
    # Benchmark
    mrr_scores = []
    for test in RETRIEVAL_BENCHMARK:
        query = query_prefix + test["query"]
        query_emb = model.encode([query], normalize_embeddings=True).tolist()
        results = collection.query(query_embeddings=query_emb, n_results=10, where={"vendor": test["vendor"]})
        mrr_scores.append(calculate_mrr(test["expected_id"], results["ids"][0]))
        
    return {"mrr": np.mean(mrr_scores)}

EMBED_MODELS = [
    "BAAI/bge-small-en-v1.5",
    "Alibaba-NLP/gte-modernbert-base",
    "ibm-granite/granite-embedding-311m-multilingual-r2",
    "answerdotai/ModernBERT-base-veblen-retrieval"
]

embed_results = {}
for m in EMBED_MODELS:
    embed_results[m] = run_embedding_test(m)

table = Table(title="Embedding Model Comparison")
table.add_column("Model", style="cyan")
table.add_column("Mean MRR", justify="right", style="green")
for model, res in embed_results.items():
    table.add_row(model, f"{res['mrr']:.4f}")
console.print(table)

## Part 2: Reranker Comparison
Uses Granite (with instructions) as the base retriever.

In [ ]:
def prepare_candidates(embed_model_name):
    console.print("\n[bold green]>>> Preparing Hybrid (BM25 + Dense) Candidates...[/bold green]")
    model = SentenceTransformer(embed_model_name, device=DEVICE, trust_remote_code=True)
    
    # 1. Setup Dense (Chroma)
    client = chromadb.EphemeralClient()
    collection = client.create_collection(name="rerank_prep", metadata={"hnsw:space": "cosine"})
    
    expected_ids = {test["expected_id"] for test in RETRIEVAL_BENCHMARK}
    relevant_chunks = [c for c in ALL_CHUNKS if c["id"] in expected_ids]
    noise_chunks = [c for c in ALL_CHUNKS if c["id"] not in expected_ids][:500]
    chunks = relevant_chunks + noise_chunks
    
    doc_prefix = "Instruction: Given a network documentation chunk, extract and embed it for retrieval.\nChunk: "
    query_prefix = "Instruction: Given a network engineering query, retrieve the most relevant command documentation.\nQuery: "
    
    embeddings = model.encode([doc_prefix + c["text"] for c in chunks], normalize_embeddings=True)
    collection.add(ids=[c["id"] for c in chunks], documents=[c["text"] for c in chunks], metadatas=[{"vendor": c["vendor"]} for c in chunks], embeddings=embeddings.tolist())

    # 2. Setup Sparse (BM25)
    tokenized_corpus = [tokenize(c["text"]) for c in chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    
    candidates = []
    found_initially = 0
    for test in RETRIEVAL_BENCHMARK:
        # A. Dense Query
        query_emb = model.encode([query_prefix + test["query"]], normalize_embeddings=True).tolist()
        dense_results = collection.query(query_embeddings=query_emb, n_results=50, where={"vendor": test["vendor"]})
        
        # B. Sparse Query
        tokenized_query = tokenize(test["query"])
        bm25_scores = bm25.get_scores(tokenized_query)
        # Filter by vendor and sort
        bm25_results = []
        for idx, score in enumerate(bm25_scores):
            if chunks[idx]["vendor"] == test["vendor"]:
                bm25_results.append((chunks[idx]["id"], chunks[idx]["text"], score))
        bm25_results = sorted(bm25_results, key=lambda x: x[2], reverse=True)[:50]
        
        # C. Reciprocal Rank Fusion (RRF)
        rrf_scores = {}
        # Dense ranks
        for rank, cid in enumerate(dense_results["ids"][0]):
            rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rank + 60)
        # BM25 ranks
        for rank, (cid, text, score) in enumerate(bm25_results):
            rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rank + 60)
        
        # Sort by RRF score
        sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:50]
        # Reconstruct doc_texts
        id_to_text = {c["id"]: c["text"] for c in chunks}
        final_texts = [id_to_text[cid] for cid in sorted_ids]
        
        is_found = test["expected_id"] in sorted_ids
        if is_found:
            found_initially += 1
        
        candidates.append({
            "query": test["query"], "expected": test["expected_id"], 
            "doc_texts": final_texts, "doc_ids": sorted_ids
        })
    
    console.print(f"[bold yellow]Hybrid Recall@50: {found_initially/len(RETRIEVAL_BENCHMARK):.2f}[/bold yellow]")
    return candidates

def run_rerank_test(model_name, candidates):
    console.print(f"[dim]Loading Reranker: {model_name}...[/dim]")
    model = CrossEncoder(model_name, device=DEVICE, trust_remote_code=True)
    
    mrr_scores = []
    inf_times = []
    for cand in candidates:
        pairs = [[cand["query"], doc] for doc in cand["doc_texts"]]
        start = time.time()
        scores = model.predict(pairs, batch_size=32, show_progress_bar=False)
        inf_times.append(time.time() - start)
        
        ranked_ids = [cand["doc_ids"][i] for i in np.argsort(scores)[::-1]]
        mrr_scores.append(calculate_mrr(cand["expected"], ranked_ids))
        
    return {"mrr": np.mean(mrr_scores), "avg_inf": np.mean(inf_times)}

RERANKERS = [
    "jinaai/jina-reranker-v2-base-multilingual",
    "jinaai/jina-reranker-v3",
    "mixedbread-ai/mxbai-rerank-base-v1",
    "BAAI/bge-reranker-v2-m3",
    "Alibaba-NLP/gte-reranker-modernbert-base",
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "BAAI/bge-reranker-base"
]

cands = prepare_candidates("ibm-granite/granite-embedding-311m-multilingual-r2")
rerank_results = {}
for r in RERANKERS:
    rerank_results[r] = run_rerank_test(r, cands)

table = Table(title="Reranker Comparison")
table.add_column("Model", style="cyan")
table.add_column("Mean MRR", justify="right", style="green")
table.add_column("Avg Inf Time (s)", justify="right", style="yellow")
for model, res in rerank_results.items():
    table.add_row(model, f"{res['mrr']:.4f}", f"{res['avg_inf']:.3f}")
console.print(table)